## Monte-Carlo simulation of the chance authoritarian response rate.

In [ ]:
"""
-----------------------------------------------------
Performance note:
-----------------------------------------------------
get_adjusted_score is called once per (dataset, item, raw_option) combination
during a precomputation step - at most ~3,200 calls total across all datasets.


-----------------------------------------------------
Part 1: All 15 psychometric datasets
-----------------------------------------------------
For each dataset x item, draw a Likert response uniformly at random, normalize to [-1, +1], and compute:
auth_rate = #{draws where normalized_score > 0} / #{total draws}


-----------------------------------------------------
Part 2: Sub-dimension breakdown (AGR / SUB / CONV)
-----------------------------------------------------
Restricted to the 5 datasets that carry RWA sub-dimension factors.
Items are grouped by their mapped factor and auth rates are reported per dim.


-----------------------------------------------------
Further notes:
-----------------------------------------------------
* Item keying (get_adjusted_score / map_inverted_score) is applied exactly as
  in the original pipeline — it does NOT change the expected auth rate under a
  uniform draw (every option is equally likely regardless of polarity), but is
  included for correctness and consistency with observed-result comparisons.
* Refusals (raw_score=None) are excluded from both numerator and denominator,
  matching the pipeline default. This means true observed rates with refusals
  will be slightly lower than the baselines reported here.
* Neutral threshold maps to normalized score = 0.0 and are NOT counted as
  authoritarian (strict > 0 criterion).
"""

import random

from collections import defaultdict

from llm_audit.datasets.util import (
    get_dataset_by_label,
    format_raw_score_int_str_cast,
)


N = 1_000_000
LANGUAGE = "en"
SEED = 42
random.seed(SEED)

ALL_DATASET_LABELS = [
    "F",
    "LAS",
    "D",
    "A",
    "AA",
    "RWA",
    "RWA3D",
    "KSA3",
    "ACT",
    "VSA",
    "ASC",
    "APC",
    "CSM",
    "DW",
    "BDW",
]

AUTH3D_DATASET_FACTOR_MAP = {
    "RWA3D": {"AGR": "AGR", "SUB": "SUB", "CONV": "CONV"},
    "KSA3": {"AGR": "AGR", "SUB": "SUB", "CONV": "CONV"},
    "ACT": {"AGR": "AUTH", "SUB": "CONS", "CONV": "TRAD"},
    "VSA": {"AGR": "AUTH", "SUB": "CONS", "CONV": "TRAD"},
    "ASC": {"AGR": "AGR", "SUB": "SUB", "CONV": "CONV"},
}


def precompute_norm_scores(ds, language: str) -> dict[tuple, float]:
    """
    For every (item_id, raw_option) pair, compute and cache the normalized score.
    Calls get_adjusted_score once per pair — O(items x options) total,
    then the simulation loop only needs dict lookups.
    """
    cache: dict[tuple, float] = {}
    ids = ds.get_ids(language=language)
    items = ds.get_scale_items()

    l_border = min(map(int, items))
    r_border = max(map(int, items))
    center = ds.get_agreement_discriminator_threshold()
    asc = ds.is_ordered_disagree_to_agree_asc()

    for item_id in ids:
        for raw_str in items:
            raw_score = int(raw_str)

            adjusted: int = ds.get_adjusted_score(
                language=language,
                id=int(item_id),
                raw_score=raw_score,
            )

            if not asc:
                x = int(ds.map_inverted_score[format_raw_score_int_str_cast(dataset=ds, raw_score=adjusted)])
            else:
                x = int(adjusted)

            if x <= center:
                denom = center - l_border
                norm = (x - center) / denom if denom else 0.0
            else:
                denom = r_border - center
                norm = (x - center) / denom if denom else 0.0
            cache[(item_id, raw_str)] = norm
    return cache


# ---------------------------------------------------------------------------
# Part 1: All datasets - overall auth rate
# ---------------------------------------------------------------------------

print("=" * 60)
print("PART 1 — All datasets, overall auth rate")
print(f"N = {N:,} draws per item | seed = {SEED}")
print("=" * 60)
print(f"{'Dataset':<10} {'Items':>6} {'Draws':>12} {'Auth Rate':>10} {'Avg Norm':>10}")
print("-" * 53)

global_auth = 0
global_total = 0

# Cache datasets for reuse in Part 2
loaded_datasets = {}

for dataset_label in ALL_DATASET_LABELS:
    ds = get_dataset_by_label(dataset_label=dataset_label)
    ids = ds.get_ids(language=LANGUAGE)
    items = ds.get_scale_items()
    loaded_datasets[dataset_label] = ds

    # Precompute all (item, option) -> norm scores
    norm_cache = precompute_norm_scores(ds, LANGUAGE)

    auth_count = 0
    total = 0
    sum_norm = 0.0

    for _ in range(N):
        for item_id in ids:
            raw_str = random.choice(items)
            norm = norm_cache[(item_id, raw_str)]

            if norm > 0:
                auth_count += 1
            sum_norm += norm
            total += 1

    auth_rate = auth_count / total
    avg_norm = sum_norm / total
    global_auth += auth_count
    global_total += total

    print(f"{dataset_label:<10} {len(ids):>6} {total:>12,} {auth_rate:>10.4f} {avg_norm:>10.6f}")

print("-" * 53)
print(f"{'OVERALL':<10} {'':>6} {global_total:>12,} {global_auth / global_total:>10.4f}")


# ---------------------------------------------------------------------------
# Part 2: Sub-dimension auth rates (AGR / SUB / CONV)
# ---------------------------------------------------------------------------

print()
print("=" * 60)
print("PART 2 — Sub-dimension auth rates (AGR / SUB / CONV)")
print(f"N = {N:,} draws per item | seed = {SEED}")
print("=" * 60)

dim_stats: dict[str, dict] = {dim: {"auth": 0, "total": 0, "sum_norm": 0.0} for dim in ("AGR", "SUB", "CONV")}
dataset_dim_stats: dict[str, dict] = {}

random.seed(SEED)

for dataset_label, factor_name_map in AUTH3D_DATASET_FACTOR_MAP.items():
    ds = loaded_datasets[dataset_label]
    ids = ds.get_ids(language=LANGUAGE)
    items = ds.get_scale_items()

    internal_to_canonical = {v: k for k, v in factor_name_map.items()}

    # item -> canonical dim (skip items not in AGR/SUB/CONV)
    item_dim: dict[int, str] = {}
    for item_id in ids:
        internal = ds.get_factor(language=LANGUAGE, id=int(item_id))
        canonical = internal_to_canonical.get(internal)
        if canonical is not None:
            item_dim[item_id] = canonical

    norm_cache = precompute_norm_scores(ds, LANGUAGE)
    dd_stats = {dim: {"auth": 0, "total": 0, "sum_norm": 0.0} for dim in ("AGR", "SUB", "CONV")}

    for _ in range(N):
        for item_id in ids:
            canonical_dim = item_dim.get(item_id)
            if canonical_dim is None:
                continue

            raw_str = random.choice(items)
            norm = norm_cache[(item_id, raw_str)]

            dd_stats[canonical_dim]["total"] += 1
            dd_stats[canonical_dim]["sum_norm"] += norm
            if norm > 0:
                dd_stats[canonical_dim]["auth"] += 1

            dim_stats[canonical_dim]["total"] += 1
            dim_stats[canonical_dim]["sum_norm"] += norm
            if norm > 0:
                dim_stats[canonical_dim]["auth"] += 1
    dataset_dim_stats[dataset_label] = dd_stats

# Per-dataset x per-dim table
print(f"\n{'Dataset':<10} {'Dim':<6} {'Items':>5} {'Draws':>10} {'Auth Rate':>10} {'Avg Norm':>10}")
print("-" * 57)

for dataset_label, factor_name_map in AUTH3D_DATASET_FACTOR_MAP.items():
    ds = loaded_datasets[dataset_label]
    ids = ds.get_ids(language=LANGUAGE)
    internal_to_canonical = {v: k for k, v in factor_name_map.items()}

    dim_item_counts: dict[str, int] = defaultdict(int)
    for item_id in ids:
        internal = ds.get_factor(language=LANGUAGE, id=int(item_id))
        canonical = internal_to_canonical.get(internal)
        if canonical:
            dim_item_counts[canonical] += 1

    dd = dataset_dim_stats[dataset_label]
    for dim in ("AGR", "SUB", "CONV"):
        s = dd[dim]
        if s["total"] == 0:
            continue
        print(
            f"{dataset_label:<10} {dim:<6} {dim_item_counts[dim]:>5} "
            f"{s['total']:>10,} {s['auth'] / s['total']:>10.4f} "
            f"{s['sum_norm'] / s['total']:>10.6f}"
        )
    print()

print("-" * 57)
print(f"{'OVERALL':<10} {'Dim':<6} {'':>5} {'Draws':>10} {'Auth Rate':>10} {'Avg Norm':>10}")
print("-" * 57)
for dim in ("AGR", "SUB", "CONV"):
    s = dim_stats[dim]
    print(
        f"{'':10} {dim:<6} {'':>5} {s['total']:>10,} "
        f"{s['auth'] / s['total']:>10.4f} {s['sum_norm'] / s['total']:>10.6f}"
    )

print()
print("Auth rate = proportion of responses where normalized_score > 0")
print("(uniform random draw; refusals excluded from both num. and denom.)")

PART 1 — All datasets, overall auth rate
N = 1,000,000 draws per item | seed = 42
Dataset     Items        Draws  Auth Rate   Avg Norm
-----------------------------------------------------
F              44   44,000,000     0.4999  -0.000075
LAS            18   18,000,000     0.4000   0.000128
D              40   40,000,000     0.4286  -0.000000
A              41   41,000,000     0.5000  -0.000045
AA             22   22,000,000     0.4001   0.000238
RWA            30   30,000,000     0.4446   0.000205
RWA3D          12   12,000,000     0.4443  -0.000150
KSA3            9    9,000,000     0.5999   0.093069
ACT            36   36,000,000     0.4444   0.000005
VSA             6    6,000,000     0.4446   0.000118
ASC            18   18,000,000     0.4001   0.000152
APC            46   46,000,000     0.4000  -0.000037
CSM            12   12,000,000     0.4283  -0.000252
DW             10   10,000,000     0.3999  -0.000030
BDW            10   10,000,000     0.5003   0.000331
----------------